In [ ]:
from google.colab import drive
drive.mount('/content/drive')
exec(open("/content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier/00_colab_setup.py").read())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ /content/drive/MyDrive/imdb_peft_project
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/roberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/deberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/oof_predictions
✓ /content/drive/MyDrive/imdb_peft_project/results
✓ /content/drive/MyDrive/imdb_peft_project/notebooks
✓ /content/drive/MyDrive/imdb_peft_project/code

Folder structure ready.
Enter GitHub Token: ··········
Repository exists, pulling latest changes...
✓ Pull complete.

Repository path: /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier
SETUP COMPLETE
Drive folder  : /content/drive/MyDrive/imdb_peft_project
GitHub repo   : /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier

Available functions:
  push_to_github('message')        → push code to GitHub
  save_code_to_rep

In [ ]:
!pip install transformers peft accelerate torchao safetensors scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import safetensors.torch as st
from transformers import (RobertaTokenizer, RobertaForSequenceClassification,
                          DebertaV2Tokenizer, DebertaV2ForSequenceClassification,
                          Trainer, TrainingArguments)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
# Define paths
DIRS["checkpoints_v2"]    = f"{DIRS['root']}/checkpoints/roberta_lora_v2"
DIRS["checkpoints2_best"] = f"{DIRS['root']}/checkpoints/deberta_lora_v2_best"
DIRS["oof_v2"]            = f"{DIRS['root']}/oof_predictions_v2"

# Load test data
test_df = pd.read_parquet(f"{DIRS['root']}/test_df_v2.parquet")
y_test  = test_df["label"].values
print(f"Test: {len(test_df)} samples")

# Load saved test probabilities
roberta_test_probs = np.load(f"{DIRS['oof_v2']}/roberta_test_probs.npy")
deberta_test_probs = np.load(f"{DIRS['oof_v2']}/deberta_test_probs.npy")
svm_test_probs     = np.load(f"{DIRS['oof_v2']}/tfidf_svm_test_probs.npy")
lr_test_probs      = np.load(f"{DIRS['oof_v2']}/tfidf_lr_test_probs.npy")

print(f"✓ All test probabilities loaded.")

# Ensemble predictions (LR weights from 06e)
# RoBERTa: 2.542, DeBERTa: 2.883, SVM: 2.37, LR: 1.119
from sklearn.linear_model import LogisticRegression
train_df    = pd.read_parquet(f"{DIRS['root']}/train_df_v2.parquet")
y_train     = train_df["label"].values

roberta_oof = np.load(f"{DIRS['oof_v2']}/roberta_v2_fold4.npy")
deberta_oof = np.load(f"{DIRS['root']}/oof_predictions_v2_best/deberta_v2_best_fold4.npy")
svm_oof     = np.load(f"{DIRS['oof_v2']}/tfidf_svm_oof.npy")
lr_oof      = np.load(f"{DIRS['oof_v2']}/tfidf_lr_oof.npy")

X_meta_train = np.column_stack([roberta_oof, deberta_oof, svm_oof, lr_oof])
X_meta_test  = np.column_stack([roberta_test_probs, deberta_test_probs,
                                 svm_test_probs, lr_test_probs])

lr_meta = LogisticRegression(random_state=SEED, max_iter=1000)
lr_meta.fit(X_meta_train, y_train)
ensemble_probs = lr_meta.predict_proba(X_meta_test)[:, 1]
ensemble_preds = lr_meta.predict(X_meta_test)

print(f"Ensemble accuracy: {accuracy_score(y_test, ensemble_preds):.4f}")

Test: 25000 samples
✓ All test probabilities loaded.
Ensemble accuracy: 0.9618


In [ ]:
# Find misclassified examples
misclassified_idx = np.where(ensemble_preds != y_test)[0]
print(f"Total misclassified: {len(misclassified_idx)} ({len(misclassified_idx)/len(y_test)*100:.2f}%)")

# False Positives: negative predicted as positive
fp_idx = np.where((ensemble_preds == 1) & (y_test == 0))[0]
# False Negatives: positive predicted as negative
fn_idx = np.where((ensemble_preds == 0) & (y_test == 1))[0]
print(f"False Positives: {len(fp_idx)} ({len(fp_idx)/len(y_test)*100:.2f}%)")
print(f"False Negatives: {len(fn_idx)} ({len(fn_idx)/len(y_test)*100:.2f}%)")

# Confidence of ensemble on misclassified examples
fp_confidence = ensemble_probs[fp_idx]
fn_confidence = 1 - ensemble_probs[fn_idx]

print(f"\nFalse Positive avg confidence : {fp_confidence.mean():.4f}")
print(f"False Negative avg confidence : {fn_confidence.mean():.4f}")

# Find most confident wrong predictions
fp_sorted = fp_idx[np.argsort(fp_confidence)[::-1]]
fn_sorted = fn_idx[np.argsort(fn_confidence)[::-1]]

print("\nTop 5 most confident False Positives:")
for i, idx in enumerate(fp_sorted[:5]):
    conf = ensemble_probs[idx]
    text = test_df.iloc[idx]["text_clean"][:200]
    print(f"\n[{i+1}] Confidence: {conf:.4f} | True: Negative")
    print(f"Text: {text}...")

print("\nTop 5 most confident False Negatives:")
for i, idx in enumerate(fn_sorted[:5]):
    conf = 1 - ensemble_probs[idx]
    text = test_df.iloc[idx]["text_clean"][:200]
    print(f"\n[{i+1}] Confidence: {conf:.4f} | True: Positive")
    print(f"Text: {text}...")

Total misclassified: 955 (3.82%)
False Positives: 519 (2.08%)
False Negatives: 436 (1.74%)

False Positive avg confidence : 0.8401
False Negative avg confidence : 0.8294

Top 5 most confident False Positives:

[1] Confidence: 0.9857 | True: Negative
Text: Mickey Rourke hunts Diane Lane in Elmore Leonard's Killshot It is not like Mickey Rourke ever really disappeared. He has had a steady string of appearances before he burst back on the scene. He was me...

[2] Confidence: 0.9856 | True: Negative
Text: This is definitely one of the best Kung fu movies in the history of Cinema. The screenplay is really well done (which is not often the case for this type of movies) and you can see that Chuck (in one ...

[3] Confidence: 0.9853 | True: Negative
Text: This movie was pure genius. John Waters is brilliant. It is hilarious and I am not sick of it even after seeing it about 20 times since I bought it a few months ago. The acting is great, although Rick...

[4] Confidence: 0.9852 | True: Negati

In [ ]:
# Analyze error patterns
print("="*60)
print("ERROR PATTERN ANALYSIS")
print("="*60)

# 1. Negation analysis
negation_words = ["not", "never", "no", "n't", "neither", "nor", "hardly", "barely"]
negation_in_fp = sum(1 for idx in fp_idx
                     if any(w in test_df.iloc[idx]["text_clean"].lower()
                            for w in negation_words))
negation_in_fn = sum(1 for idx in fn_idx
                     if any(w in test_df.iloc[idx]["text_clean"].lower()
                            for w in negation_words))

print(f"\nNegation words in False Positives: {negation_in_fp}/{len(fp_idx)} ({negation_in_fp/len(fp_idx)*100:.1f}%)")
print(f"Negation words in False Negatives: {negation_in_fn}/{len(fn_idx)} ({negation_in_fn/len(fn_idx)*100:.1f}%)")

# 2. Review length analysis
fp_lengths = [len(test_df.iloc[idx]["text_clean"].split()) for idx in fp_idx]
fn_lengths = [len(test_df.iloc[idx]["text_clean"].split()) for idx in fn_idx]
correct_lengths = [len(test_df.iloc[idx]["text_clean"].split())
                   for idx in range(len(test_df)) if ensemble_preds[idx] == y_test[idx]]

print(f"\nAvg review length:")
print(f"  Correctly classified : {np.mean(correct_lengths):.1f} words")
print(f"  False Positives      : {np.mean(fp_lengths):.1f} words")
print(f"  False Negatives      : {np.mean(fn_lengths):.1f} words")

# 3. Long review error rate
long_idx  = [i for i in range(len(test_df))
             if len(test_df.iloc[i]["text_clean"].split()) > 400]
short_idx = [i for i in range(len(test_df))
             if len(test_df.iloc[i]["text_clean"].split()) <= 400]

long_errors  = sum(1 for i in long_idx  if ensemble_preds[i] != y_test[i])
short_errors = sum(1 for i in short_idx if ensemble_preds[i] != y_test[i])

print(f"\nError rate for long reviews  (>400 words): {long_errors/len(long_idx)*100:.2f}%")
print(f"Error rate for short reviews (≤400 words): {short_errors/len(short_idx)*100:.2f}%")

# 4. Model disagreement analysis
roberta_preds = (roberta_test_probs > 0.5).astype(int)
deberta_preds = (deberta_test_probs > 0.5).astype(int)
disagree_idx  = np.where(roberta_preds != deberta_preds)[0]

print(f"\nModel disagreement:")
print(f"  Total disagreements      : {len(disagree_idx)} ({len(disagree_idx)/len(y_test)*100:.1f}%)")

disagree_errors = sum(1 for i in disagree_idx if ensemble_preds[i] != y_test[i])
print(f"  Errors in disagreements  : {disagree_errors} ({disagree_errors/len(disagree_idx)*100:.1f}%)")

agree_idx   = np.where(roberta_preds == deberta_preds)[0]
agree_errors = sum(1 for i in agree_idx if ensemble_preds[i] != y_test[i])
print(f"  Errors when both agree   : {agree_errors} ({agree_errors/len(agree_idx)*100:.1f}%)")

resolved = sum(1 for i in disagree_idx
               if ensemble_preds[i] == y_test[i] and
               (roberta_preds[i] != y_test[i] or deberta_preds[i] != y_test[i]))
print(f"  Disagreements resolved by ensemble: {resolved} ({resolved/len(disagree_idx)*100:.1f}%)")

ERROR PATTERN ANALYSIS

Negation words in False Positives: 474/519 (91.3%)
Negation words in False Negatives: 416/436 (95.4%)

Avg review length:
  Correctly classified : 226.0 words
  False Positives      : 223.6 words
  False Negatives      : 240.5 words

Error rate for long reviews  (>400 words): 4.05%
Error rate for short reviews (≤400 words): 3.79%

Model disagreement:
  Total disagreements      : 699 (2.8%)
  Errors in disagreements  : 246 (35.2%)
  Errors when both agree   : 709 (2.9%)
  Disagreements resolved by ensemble: 453 (64.8%)


In [ ]:
# Select interesting examples for the report

print("EXAMPLE 1 — Highly confident False Positive (sarcasm/irony):")
idx = fp_sorted[2]  # "pure genius" example
print(f"True label: Negative | Ensemble confidence: {ensemble_probs[idx]:.4f}")
print(f"RoBERTa: {roberta_test_probs[idx]:.4f} | DeBERTa: {deberta_test_probs[idx]:.4f}")
print(f"\nText (first 300 chars):")
print(test_df.iloc[idx]["text_clean"][:300])

print("\n" + "="*60)

print("\nEXAMPLE 2 — Highly confident False Negative (negative words in positive review):")
idx = fn_sorted[0]  # SPOILERS example
print(f"True label: Positive | Ensemble confidence: {1-ensemble_probs[idx]:.4f}")
print(f"RoBERTa: {roberta_test_probs[idx]:.4f} | DeBERTa: {deberta_test_probs[idx]:.4f}")
print(f"\nText (first 300 chars):")
print(test_df.iloc[idx]["text_clean"][:300])

print("\n" + "="*60)

print("\nEXAMPLE 3 — Model disagreement example:")
# Find a disagreement where RoBERTa is right but DeBERTa is wrong
for idx in disagree_idx:
    if roberta_preds[idx] == y_test[idx] and deberta_preds[idx] != y_test[idx]:
        print(f"True label: {y_test[idx]} | RoBERTa: {roberta_test_probs[idx]:.4f} | DeBERTa: {deberta_test_probs[idx]:.4f}")
        print(f"Ensemble: {ensemble_probs[idx]:.4f}")
        print(f"\nText (first 300 chars):")
        print(test_df.iloc[idx]["text_clean"][:300])
        break

EXAMPLE 1 — Highly confident False Positive (sarcasm/irony):
True label: Negative | Ensemble confidence: 0.9853
RoBERTa: 0.9981 | DeBERTa: 0.9987

Text (first 300 chars):
This movie was pure genius. John Waters is brilliant. It is hilarious and I am not sick of it even after seeing it about 20 times since I bought it a few months ago. The acting is great, although Ricki Lake could have been better. And Johnny Depp is magnificent. He is such a beautiful man and a very


EXAMPLE 2 — Highly confident False Negative (negative words in positive review):
True label: Positive | Ensemble confidence: 0.9896
RoBERTa: 0.0020 | DeBERTa: 0.0024

Text (first 300 chars):
**SPOILERS AHEAD** It is really unfortunate that a movie so well produced turns out to be such a disappointment. I thought this was full of (silly) clichés and that it basically tried to hard. To the (American) guys out there: how many of you spend your time jumping on your girlfriend's bed and maki


EXAMPLE 3 — Model disagreement e

In [ ]:
# Save error analysis results
save_results({
    "total_misclassified": 955,
    "false_positives": 519,
    "false_negatives": 436,
    "fp_rate": 2.08,
    "fn_rate": 1.74,
    "negation_in_fp": 91.3,
    "negation_in_fn": 95.4,
    "long_review_error_rate": 4.05,
    "short_review_error_rate": 3.79,
    "total_disagreements": 699,
    "disagreement_rate": 2.8,
    "error_rate_on_disagreements": 35.2,
    "ensemble_resolved_disagreements": 64.8,
    "examples": {
        "false_positive_irony": {
            "text": "This movie was pure genius. John Waters is brilliant...",
            "roberta_conf": 0.9981,
            "deberta_conf": 0.9987,
            "ensemble_conf": 0.9853
        },
        "false_negative_misleading": {
            "text": "**SPOILERS AHEAD** It is really unfortunate...",
            "roberta_conf": 0.0020,
            "deberta_conf": 0.0024,
            "ensemble_conf": 0.9896
        },
        "model_disagreement": {
            "text": "This is a hard film to rate...",
            "roberta_conf": 0.0618,
            "deberta_conf": 0.8106,
            "ensemble_conf": 0.1605
        }
    }
}, "error_analysis_results.json")

NOTEBOOK_NAME = "error_analysis"
!jupyter nbconvert --to script \
  "/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}.ipynb" \
  --output-dir "/content/"
import os
os.rename(f"/content/{NOTEBOOK_NAME}.txt",
          f"/content/{NOTEBOOK_NAME}.py")
save_code_to_repo(f"/content/{NOTEBOOK_NAME}.py")
push_to_github("error analysis complete with real examples")

✓ Results saved: /content/drive/MyDrive/imdb_peft_project/results/error_analysis_results.json
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/error_analysis.ipynb to script
[NbConvertApp] Writing 8486 bytes to /content/error_analysis.txt
✓ error_analysis.py → copied to repository.
✓ Pushed to GitHub: 'error analysis complete with real examples'
